In [19]:
import requests
from bs4 import BeautifulSoup
from typing import List, Dict
import re
import time

# -----------------------------
# 1. Fetch HTML
# -----------------------------
def fetch_page(url: str) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; TripReportBot/1.0)"
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.text


# -----------------------------
# 2. Parse posts from page
# -----------------------------
def parse_posts(html: str) -> List[Dict]:
    soup = BeautifulSoup(html, "html.parser")

    posts = []

    # High Sierra Topix uses "postbody" class for content
    post_bodies = soup.find_all("div", class_="postbody")

    for post in post_bodies:
        content_div = post.find("div", class_="content")

        if not content_div:
            continue

        text = content_div.get_text(separator="\n", strip=True)

        posts.append({
            "text": text
        })

    return posts


# -----------------------------
# 3. Filter for Wind River mentions
# -----------------------------
def filter_fish(posts: List[Dict]) -> List[Dict]:
    pattern = re.compile(r"\b(trout|fish)\b", re.IGNORECASE)

    filtered = []
    for post in posts:
        if pattern.search(post["text"]):
            filtered.append(post)

    return filtered
    
def filter_wind_river(posts: List[Dict]) -> List[Dict]:
    pattern = re.compile(r"\b(wind rivers?|the winds)\b", re.IGNORECASE)

    filtered = []
    for post in posts:
        if pattern.search(post["text"]):
            filtered.append(post)

    return filtered

def filter_sierras(posts: List[Dict]) -> List[Dict]:
    pattern = re.compile(r"\bsierra?\b", re.IGNORECASE)

    filtered = []
    for post in posts:
        if pattern.search(post["text"]):
            filtered.append(post)

    return filtered

# -----------------------------
# 4. Main pipeline
# -----------------------------
def scrape_trip_report(url: str) -> List[Dict]:
    html = fetch_page(url)
    posts = parse_posts(html)
    fish_posts = filter_fish(posts)
    wind_posts = filter_wind_river(fish_posts)
    sierras_posts = filter_sierras(fish_posts)

    return wind_posts, sierras_posts

In [37]:
import requests

def fetch_page(url: str) -> str:
    headers = {"User-Agent": "TripCrawler/1.0"}
    res = requests.get(url, headers=headers)
    res.raise_for_status()
    return res.text

from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://www.highsierratopix.com/community/"

def parse_thread_links(html: str) -> list:
    soup = BeautifulSoup(html, "html.parser")
    links = []

    for a in soup.find_all("a", href=True):
        href = a["href"]

        if "viewtopic.php?t=" in href:
            full_url = urljoin(BASE_URL, href)
            links.append(full_url)

    return list(set(links))  # dedupe

import json
import os

SEEN_FILE = "seen_threads.json"

def load_seen():
    if not os.path.exists(SEEN_FILE):
        return set()
    return set(json.load(open(SEEN_FILE)))

def save_seen(seen):
    json.dump(list(seen), open(SEEN_FILE, "w"))

def get_new_threads(links, seen):
    new_links = [l for l in links if l not in seen]
    return new_links

def crawl_forum(start=0, stop=100):
    seen = load_seen()
    all_new = []
    info = []
    for offset in range(start, stop, 25):  # adjust range as needed
        url = f"https://www.highsierratopix.com/community/viewforum.php?f=1&start={offset}"

        html = fetch_page(url)
        time.sleep(2)
        links = parse_thread_links(html)
        print(links)

        new_links = get_new_threads(links, seen)
                
        print(f"Found {len(new_links)} new threads on page {offset}")

        all_new.extend(new_links)
        seen.update(new_links)

    save_seen(seen)
    return all_new

In [39]:
new_links = crawl_forum(400, 600)



['https://www.highsierratopix.com/community/viewtopic.php?t=24001&start=10', 'https://www.highsierratopix.com/community/viewtopic.php?t=23194', 'https://www.highsierratopix.com/community/viewtopic.php?t=20478&start=10', 'https://www.highsierratopix.com/community/viewtopic.php?t=23258&start=10', 'https://www.highsierratopix.com/community/viewtopic.php?t=23284', 'https://www.highsierratopix.com/community/viewtopic.php?t=23386&start=10', 'https://www.highsierratopix.com/community/viewtopic.php?t=23348&start=10', 'https://www.highsierratopix.com/community/viewtopic.php?t=23372&start=10', 'https://www.highsierratopix.com/community/viewtopic.php?t=22491', 'https://www.highsierratopix.com/community/viewtopic.php?t=23386', 'https://www.highsierratopix.com/community/viewtopic.php?t=23385', 'https://www.highsierratopix.com/community/viewtopic.php?t=23368', 'https://www.highsierratopix.com/community/viewtopic.php?t=1431', 'https://www.highsierratopix.com/community/viewtopic.php?t=23351', 'https:/

In [40]:
len(new_links)


299

In [41]:
info = []
from tqdm import tqdm

for i, url in tqdm(enumerate(new_links)):
    # print(f'checking URL:{url}')
    winds, sierras = scrape_trip_report(url)

    for i, post in enumerate(winds, 1):
        print(f"\n--- Post {i} ---\n")
        print(post["text"][:500])  # preview
        info.append({"source_url": url,
        "text": post["text"],
        "region_hint": "wind rivers"})

    for i, post in enumerate(sierras, 1):
        print(f"\n--- Post {i} ---\n")
        print(post["text"][:500])  # preview
        info.append({"source_url": url,
        "text": post["text"],
        "region_hint": "sierra nevada"})
    time.sleep(5)

3it [00:16,  5.56s/it]


--- Post 1 ---

Hi All,
I alluded to this matter in a post in the Campfire section, but for greater visibility, I thought it would be best to alert folk here. Specifically, on a recent trip in which my wife and I transited Grinnell-Hopkins ridge, there was a piece of equipment that we inferred was a motion-activated camera (and it included the indication that it was placed there by the Department of Fish and Wildlife; DFW). I finally touched base with DFW and promptly heard back from Brian Hatfield, the individ


19it [01:44,  5.45s/it]


--- Post 1 ---

Heya,
I'm tacking on a high Sierra trip onto a work trip out west in early October (juggling shifting weather and a potential gov shutdown).  I'm a crusty old thru hiker from the 90s. I do a ton of solo, multi night trips out in the Pisgah, Nantahala, Unicoi out in TN/NC. But it's been a long time since I've been above 9k ft.
I've got a lot of flexibility. Perfect plan is 3 nights up Piute Pass, over Humphreys Basin and Puppet Pass, down French Canyon and back over Piute Pass. Plan to acclimate 


25it [02:17,  5.45s/it]

...

262it [24:01,  5.50s/it]


--- Post 1 ---

Fourth Recess Lake, from Rock Creek, via Mono Pass. (Note there are two passes named Mono Pass; this is the one furthest south, above Ruby Lake.)  Pack UL style and this is a good trip for the time allotted, but can be a schlep with a more thoroughly equipped kit if you are not in top condition.  You will have some company at the lake, but it is huge, with plenty of suitable sites, so it is easy to find ample elbow room.  Most of the route passes through alpine terrain, but Fourth Recess basin h


278it [25:29,  5.46s/it]


--- Post 1 ---

jefffish
wrote:
↑
Wed Aug 03, 2022 8:13 am
Interesting choice Jon. I've always wondered if there were fish in Warren. So really looking forward to your report. Last I heard (10yrs ago) it was overrun by golden shiners and very few trout, if any. Have never really heard anything about Devil's Oven Lake or Paradise either, fishing-wise. Good luck.
~Jeff
Given the timing of this planned trip in mid August it's fair to say that it is unlikely that this will be a fair test of the quality of the fishe


299it [27:23,  5.50s/it]


In [42]:
len(info)

23

In [43]:
from openai import OpenAI
import json
import os
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)

def parse_trip_report(text: str) -> dict:
    prompt = f"""
Extract structured fish observations from this trip report. There may be multiple fish observations.
Return a single observation for every lake-species-date pair. Only lake_name and species are required.

Return ONLY valid JSON with a list of objects of the following form:
- lake_name
- species
- count (int/null)
- length (int/null)
- date
- notes

Trip report:
{text}
"""

    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt,
    )

    output_text = response.output[0].content[0].text
    return output_text
    # try:
    #     print(output_text)
    #     return json.loads(output_text)
    # except json.JSONDecodeError:
    #     print("Failed to parse JSON")
    #     return None
responses = []
for post in info:
    responses.append(parse_trip_report(info[0]["text"]))
    # stripped = response[response.find("["): response.rfi

In [44]:
responses

['```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```',
 '```json\n[]\n```']